# Download structured document artifacts by ID

This notebook downloads every object under:

`gs://court_data_2024_structured/document_text_parsing/info_version_9/2/{doc_id}/`

The remote folder structure is preserved inside the local `downloads/{doc_id}/` directory. Authentication uses Google Application Default Credentials.

In [ ]:
# Colab only:
# from google.colab import auth
# auth.authenticate_user()
# %pip install -q google-cloud-storage

In [2]:
from pathlib import Path

from google.cloud import storage

PROJECT_ID = "lab-test-project-1-305710"
BUCKET_NAME = "court_data_2024_structured"
ROOT_PREFIX = "document_text_parsing/info_version_9"
JUSTICE_KIND = 2
OUTPUT_DIRECTORY = Path.cwd() / "downloads"

## Choose the document

In [7]:
DOCUMENT_ID = "118355359"  # Replace with the required document ID

## Download all files in the document folder

In [8]:
def download_document_folder(
    document_id: str | int,
    output_directory: Path,
) -> list[Path]:
    document_id = str(document_id).strip()
    if not document_id.isdigit():
        raise ValueError("document_id must contain digits only")

    prefix = f"{ROOT_PREFIX}/{JUSTICE_KIND}/{document_id}/"
    destination_root = output_directory / document_id
    client = storage.Client(project=PROJECT_ID)
    blobs = list(client.list_blobs(BUCKET_NAME, prefix=prefix))
    file_blobs = [blob for blob in blobs if not blob.name.endswith("/")]

    if not file_blobs:
        raise FileNotFoundError(f"No files found at gs://{BUCKET_NAME}/{prefix}")

    downloaded_paths = []
    for blob in file_blobs:
        relative_path = Path(blob.name.removeprefix(prefix))
        destination = destination_root / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(str(destination))
        downloaded_paths.append(destination)
        print(f"Downloaded {blob.name} -> {destination.resolve()}")

    print(f"\nDownloaded {len(downloaded_paths)} files to {destination_root.resolve()}")
    return downloaded_paths


downloaded_paths = download_document_folder(DOCUMENT_ID, OUTPUT_DIRECTORY)

Downloaded document_text_parsing/info_version_9/2/118355359/_document.json -> /Users/vegura/Projects/university/legal_doc_parser/src/document_split/v2/document_text_parsing/downloads/118355359/_document.json
Downloaded document_text_parsing/info_version_9/2/118355359/classification.parquet -> /Users/vegura/Projects/university/legal_doc_parser/src/document_split/v2/document_text_parsing/downloads/118355359/classification.parquet
Downloaded document_text_parsing/info_version_9/2/118355359/numbered_document.json -> /Users/vegura/Projects/university/legal_doc_parser/src/document_split/v2/document_text_parsing/downloads/118355359/numbered_document.json
Downloaded document_text_parsing/info_version_9/2/118355359/paragraphs.parquet -> /Users/vegura/Projects/university/legal_doc_parser/src/document_split/v2/document_text_parsing/downloads/118355359/paragraphs.parquet

Downloaded 4 files to /Users/vegura/Projects/university/legal_doc_parser/src/document_split/v2/document_text_parsing/downloads/

## Display the downloaded result with pandas

The first table lists the downloaded artifacts. The second displays every row from `paragraphs.parquet`.

In [10]:
import pandas as pd
from IPython.display import display

files_df = pd.DataFrame(
    {
        "file_name": [path.name for path in downloaded_paths],
        "local_path": [str(path.resolve()) for path in downloaded_paths],
        "size_bytes": [path.stat().st_size for path in downloaded_paths],
    }
)
display(files_df)

paragraphs_path = next(
    (path for path in downloaded_paths if path.name == "classification.parquet"),
    None,
)
if paragraphs_path is None:
    raise FileNotFoundError("paragraphs.parquet was not found in the downloaded folder")

paragraphs_df = pd.read_parquet(paragraphs_path)
print(f"Paragraph rows: {len(paragraphs_df):,}")
display(paragraphs_df)

,file_name,local_path,size_bytes
0,_document.json,/Users/vegura/Projects/university/legal_doc_pa...,181
1,classification.parquet,/Users/vegura/Projects/university/legal_doc_pa...,14288
2,numbered_document.json,/Users/vegura/Projects/university/legal_doc_pa...,19152
3,paragraphs.parquet,/Users/vegura/Projects/university/legal_doc_pa...,14048


Paragraph rows: 41


,document_id,paragraph_index,paragraph_order,numbered_text,text,section
0,118355359,1,1,[paragraph_id=1] Справа № 369/9945/16-к,Справа № 369/9945/16-к,introductory
1,118355359,2,2,[paragraph_id=2] Провадження № 1-кп/369/83/24,Провадження № 1-кп/369/83/24,introductory
2,118355359,3,3,[paragraph_id=3] В И Р О К,В И Р О К,introductory
3,118355359,4,4,[paragraph_id=4] іменем України,іменем України,introductory
4,118355359,5,5,[paragraph_id=5] 15.04.24 року м. Київ,15.04.24 року м. Київ,introductory
5,118355359,6,6,[paragraph_id=6] Києво-Святошинський районний ...,Києво-Святошинський районний суд Київської обл...,introductory
6,118355359,7,7,[paragraph_id=7] головуючого судді: ОСОБА_1,головуючого судді: ОСОБА_1,introductory
7,118355359,8,8,[paragraph_id=8] при секретарці: ОСОБА_2,при секретарці: ОСОБА_2,introductory
8,118355359,9,9,[paragraph_id=9] за участі прокурорів: ОСОБА_3...,"за участі прокурорів: ОСОБА_3 , ОСОБА_4",introductory
9,118355359,10,10,"[paragraph_id=10] захисників ОСОБА_5 , ОСОБА_6 ,","захисників ОСОБА_5 , ОСОБА_6 ,",introductory
